# Benchmark: EN→CY Translation

Compares our model against Opus-MT, NLLB-200 (distilled 600M and 1.3B), and Small-100 on multiple datasets using BLEU, chrF, and chrF++.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import sacrebleu
import yaml
import pandas as pd
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer, AutoModelForSeq2SeqLM, AutoTokenizer

from src import datasets as ds_module, utils
from src.hf_wrapper import EnCyForTranslation
from src.tokenizer import load_tokenizer

CONFIG = "base"
NUM_SENTENCES = 1000
BATCH_SIZE = 32

with open(f"../configs/{CONFIG}.yaml") as f:
    config = yaml.safe_load(f)

run_path = utils.get_run_path(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load datasets

In [ ]:
# Cardiff test split
ds = ds_module.get_dataset("train", config)
cardiff_test = ds["test"].select(range(min(NUM_SENTENCES, len(ds["test"]))))
print(f"Cardiff test: {len(cardiff_test)} sentences")

# FLORES+ devtest
flores = load_dataset("openlanguagedata/flores_plus")["devtest"]
flores_eng = flores.filter(lambda x: x["iso_639_3"] == "eng")
flores_cym = flores.filter(lambda x: x["iso_639_3"] == "cym")
print(f"FLORES+ devtest: {len(flores_eng)} sentences")

# Tatoeba
tatoeba = load_dataset("agentlans/tatoeba-english-translations")["train"]
tatoeba_cy = tatoeba.filter(lambda x: x["Language"] == "cym")
print(f"Tatoeba: {len(tatoeba_cy)} sentences")

eval_datasets = {
    "Cardiff": {"src": cardiff_test["text_en"], "ref": cardiff_test["text_cy"]},
    "FLORES+": {"src": flores_eng["text"], "ref": flores_cym["text"]},
    "Tatoeba": {"src": tatoeba_cy["English"], "ref": tatoeba_cy["Translation"]},
}

## Helpers

In [ ]:
def translate_batch(model, tokenizer, sentences, max_length=256, src_lang=None, tgt_lang=None, batch_size=BATCH_SIZE):
    model.eval()
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        if src_lang:
            tokenizer.src_lang = src_lang
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
        inputs.pop("token_type_ids", None)
        inputs = inputs.to(device)
        gen_kwargs = {"max_length": max_length, "num_beams": 4}
        if tgt_lang:
            gen_kwargs["forced_bos_token_id"] = tokenizer.convert_tokens_to_ids(tgt_lang)
        with torch.no_grad():
            output_ids = model.generate(**inputs, **gen_kwargs)
        results.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
    return results


def score(hypotheses, references):
    bleu = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="intl")
    spbleu = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="flores200")
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=0)
    chrfpp = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    return {
        "BLEU": round(bleu.score, 2),
        "spBLEU": round(spbleu.score, 2),
        "chrF": round(chrf.score, 2),
        "chrF++": round(chrfpp.score, 2),
    }

## Run all models

In [ ]:
all_results = {}
all_hyps = {}

# --- Our model ---
print("Loading our model...")
our_model = EnCyForTranslation.from_run(run_path).to(device)
our_tok = load_tokenizer(run_path)
for ds_name, ds_data in eval_datasets.items():
    print(f"  Translating {ds_name}...")
    hyps = translate_batch(our_model, our_tok, ds_data["src"], max_length=config["model"]["max_length"])
    all_results[("Ours", ds_name)] = score(hyps, ds_data["ref"])
    all_hyps[("Ours", ds_name)] = hyps
del our_model
torch.cuda.empty_cache()
print("Done.")

# --- Opus-MT ---
print("Loading Opus-MT...")
opus_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-cy")
opus_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-cy").to(device)
for ds_name, ds_data in eval_datasets.items():
    print(f"  Translating {ds_name}...")
    hyps = translate_batch(opus_model, opus_tok, ds_data["src"])
    all_results[("Opus-MT", ds_name)] = score(hyps, ds_data["ref"])
    all_hyps[("Opus-MT", ds_name)] = hyps
del opus_model
torch.cuda.empty_cache()
print("Done.")

# --- NLLB-200 (distilled 600M) ---
print("Loading NLLB-200 600M...")
nllb_tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M").to(device)
for ds_name, ds_data in eval_datasets.items():
    print(f"  Translating {ds_name}...")
    hyps = translate_batch(nllb_model, nllb_tok, ds_data["src"], src_lang="eng_Latn", tgt_lang="cym_Latn")
    all_results[("NLLB-200 600M", ds_name)] = score(hyps, ds_data["ref"])
    all_hyps[("NLLB-200 600M", ds_name)] = hyps
del nllb_model
torch.cuda.empty_cache()
print("Done.")

# --- NLLB-200 (distilled 1.3B) ---
# Loaded in fp16 with a small batch to keep GPU load down on an 8 GB laptop card
print("Loading NLLB-200 1.3B...")
nllb13_tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-1.3B")
nllb13_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-1.3B", torch_dtype=torch.float16).to(device)
for ds_name, ds_data in eval_datasets.items():
    print(f"  Translating {ds_name}...")
    hyps = translate_batch(nllb13_model, nllb13_tok, ds_data["src"], src_lang="eng_Latn", tgt_lang="cym_Latn", batch_size=4)
    all_results[("NLLB-200 1.3B", ds_name)] = score(hyps, ds_data["ref"])
    all_hyps[("NLLB-200 1.3B", ds_name)] = hyps
del nllb13_model
torch.cuda.empty_cache()
print("Done.")

# --- Small-100 ---
print("Loading Small-100...")
sm100_tok = AutoTokenizer.from_pretrained("alirezamsh/small100")
sm100_model = AutoModelForSeq2SeqLM.from_pretrained("alirezamsh/small100").to(device)
for ds_name, ds_data in eval_datasets.items():
    print(f"  Translating {ds_name}...")
    hyps = translate_batch(sm100_model, sm100_tok, ds_data["src"], tgt_lang="cy")
    all_results[("Small-100", ds_name)] = score(hyps, ds_data["ref"])
    all_hyps[("Small-100", ds_name)] = hyps
del sm100_model
torch.cuda.empty_cache()
print("Done.")

## Results

In [ ]:
rows = []
for (model_name, ds_name), scores in all_results.items():
    rows.append({"Model": model_name, "Dataset": ds_name, **scores})

df = pd.DataFrame(rows)

for ds_name in eval_datasets:
    print(f"\n=== {ds_name} ===")
    subset = df[df["Dataset"] == ds_name].set_index("Model")[["BLEU", "spBLEU", "chrF", "chrF++"]].sort_values("BLEU", ascending=False)
    display(subset)

## Example translations

In [ ]:
models = ["Ours", "Opus-MT", "NLLB-200 600M", "NLLB-200 1.3B", "Small-100"]
ds_name = "FLORES+"

N = 10
src = eval_datasets[ds_name]["src"]
ref = eval_datasets[ds_name]["ref"]
for i in range(N):
    print(f"EN:        {src[i]}")
    print(f"Reference: {ref[i]}")
    for m in models:
        print(f"{m + ':':16s}{all_hyps[(m, ds_name)][i]}")
    print()